In [8]:
import json
import os
import zipfile
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)

ZIP_PATH = Path("data/raw/data-056a457e-63ac-4929-8230-981eb31cce48-1780348966-1ccdf4ae-batch-0000.zip")

In [ ]:
%load_ext sql
import duckdb
_conn = duckdb.connect("data/ai_chat.duckdb", read_only=True)
%sql _conn

In [18]:
%%sql
SELECT * FROM main_staging.stg_messages LIMIT 5;

Running query in 'duckdb:///data/ai_chat.duckdb'

message_id,conversation_id,sender,text,created_at,updated_at,parent_message_id,has_attachments,has_files,text_length
778e2790-ccd7-44cf-9858-4e3e355dcc21,6b6afe41-ab0d-4b0e-8f39-44a18c80a10b,human,hello how are you doing,2024-10-24 10:13:38.843082+01:00,2024-10-24 10:13:38.843082+01:00,00000000-0000-4000-8000-000000000000,False,False,23
05780cb3-8795-4857-babe-83d38facf9d3,6b6afe41-ab0d-4b0e-8f39-44a18c80a10b,assistant,"Hi! I'm doing well, thank you for asking. How are you today?",2024-10-24 10:13:38.843082+01:00,2024-10-24 10:13:38.843082+01:00,778e2790-ccd7-44cf-9858-4e3e355dcc21,False,False,61
306bd439-1f26-485e-aff6-6c7ce689858f,6b6afe41-ab0d-4b0e-8f39-44a18c80a10b,human,Can you write a really long message,2024-10-24 10:13:57.952936+01:00,2024-10-24 10:13:57.952936+01:00,05780cb3-8795-4857-babe-83d38facf9d3,False,False,35
6d08a2bc-eeec-4e11-8cc0-e32e125ad89e,6b6afe41-ab0d-4b0e-8f39-44a18c80a10b,assistant,"I aim to be helpful while keeping responses focused and concise. Rather than writing a very long message just for the sake of length, I'd be happy to explore a specific topic that interests you in appropriate detail. What subject would you like to discuss or learn more about?",2024-10-24 10:13:57.952936+01:00,2024-10-24 10:13:57.952936+01:00,306bd439-1f26-485e-aff6-6c7ce689858f,False,False,277
5804db08-89ce-421f-a2ce-5a3f5c86ad68,0a9802a9-a708-4cd7-808e-63978115af62,human,Can you generate some boilerplate react ts vite code for a chrome extension,2024-10-25 19:21:02.439211+01:00,2024-10-25 19:21:02.439211+01:00,00000000-0000-4000-8000-000000000000,False,False,75


In [16]:
%%sql
SELECT * FROM main_staging.stg_conversations
WHERE 
    summary != ''
LIMIT 2;

Running query in 'duckdb:///data/ai_chat.duckdb'

conversation_id,conversation_name,summary,created_at,updated_at,account_id
9c68496c-261a-4a0c-a42a-6a43e4736ae1,Stocks and shares ISA balance not updating,"**Conversation Overview**The person asked two questions about investing using InvestEngine's Stocks and Shares ISA. First, they wanted to understand why their balance wasn't changing after depositing money. Claude explained that funds deposited into an InvestEngine ISA sit as uninvested cash until the user actively purchases ETFs, and that cash held in the account does not earn returns. Claude outlined the distinction between account cash and portfolio cash, described the DIY portfolio option involving selection from approximately 600 ETFs, and mentioned the automated savings plan feature for recurring investments.Second, the person asked how to transfer their Stocks and Shares ISA from InvestEngine to Trading 212. Claude explained that the transfer must be initiated from Trading 212 as the receiving provider, walked through the steps within the Trading 212 app under ""Portfolio transfers,"" and described the two transfer methods available: a cash transfer (investments sold, cash moved, faster but leaves the person out of the market temporarily) and an in specie transfer (holdings move without being sold, though fractional shares must be liquidated, taking approximately four to six weeks). Claude noted that the annual ISA allowance is unaffected by transfers and that Trading 212 does not charge transfer fees, while advising the person to check whether InvestEngine has any exit fees.",2026-02-01 14:16:50.840472+00:00,2026-02-01 14:38:47.918999+00:00,0fdaf96e-1ded-444b-8597-acf56a513afc
9be821a2-f8b1-4870-a7e7-425cea3df1f0,Screenshot organization browser extension,"**Conversation Overview**The person is exploring building a browser extension focused on screenshot capture and folder-based organization. The conversation covered whether similar tools already exist, with Claude noting that while desktop apps like Lightshot and ShareX and browser extensions like Awesome Screenshot exist, there is a gap in the market around thoughtful local organization rather than annotation or sharing. Key open questions about the project include whether it would use local or cloud storage, whether it would capture web content only or full-screen, and potential features like OCR or duplicate detection.A significant portion of the conversation focused on naming the extension. The person initially suggested ""Photo Pocket"" but recognized it would perform poorly for SEO. Claude walked through naming strategies, distinguishing between purely descriptive names (better for discoverability) and brandable names paired with descriptive subtitles. The person expressed interest in ""SnapStash,"" which Claude researched and found to be associated with an existing physical product in a niche consumer category and a photo-organization app called SnapStash.ai, leading the person to decide they wanted something more distinct. Claude then offered a further round of name suggestions including Screenie, SnapFiler, Clippet, Snapdex, and ScreenKeep, among others. The conversation ended without a final name decision. The person's stated preference is for a name that is distinct and memorable while still supporting SEO, and they are open to creative options rather than purely generic descriptive names.",2026-02-01 14:49:32.043375+00:00,2026-02-01 14:53:49.053469+00:00,0fdaf96e-1ded-444b-8597-acf56a513afc


In [2]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    conversations = json.loads(zf.read('conversations.json'))

print(f'{len(conversations)} conversations')

348 conversations


## Flatten into messages

In [3]:
rows = []
for conv in conversations:
    for msg in conv.get('chat_messages', []):
        rows.append({
            'conversation_uuid': conv['uuid'],
            'conversation_name': conv.get('name'),
            'conversation_created_at': conv.get('created_at'),
            'message_uuid': msg['uuid'],
            'sender': msg['sender'],
            'text': msg.get('text', ''),
            'created_at': msg['created_at'],
            'parent_message_uuid': msg.get('parent_message_uuid'),
            'has_attachments': len(msg.get('attachments', [])) > 0,
            'has_files': len(msg.get('files', [])) > 0,
        })

df = pd.DataFrame(rows)
df['created_at'] = pd.to_datetime(df['created_at'])
df['conversation_created_at'] = pd.to_datetime(df['conversation_created_at'])
df['text_length'] = df['text'].str.len()

print(df.shape)
df.head()

(2698, 11)


,conversation_uuid,conversation_name,conversation_created_at,message_uuid,sender,text,created_at,parent_message_uuid,has_attachments,has_files,text_length
0,6b6afe41-ab0d-4b0e-8f39-44a18c80a10b,Friendly Greeting,2024-10-24 10:13:36.033454+00:00,778e2790-ccd7-44cf-9858-4e3e355dcc21,human,hello how are you doing,2024-10-24 10:13:38.843082+00:00,00000000-0000-4000-8000-000000000000,False,False,23
1,6b6afe41-ab0d-4b0e-8f39-44a18c80a10b,Friendly Greeting,2024-10-24 10:13:36.033454+00:00,05780cb3-8795-4857-babe-83d38facf9d3,assistant,"Hi! I'm doing well, thank you for asking. How...",2024-10-24 10:13:38.843082+00:00,778e2790-ccd7-44cf-9858-4e3e355dcc21,False,False,61
2,6b6afe41-ab0d-4b0e-8f39-44a18c80a10b,Friendly Greeting,2024-10-24 10:13:36.033454+00:00,306bd439-1f26-485e-aff6-6c7ce689858f,human,Can you write a really long message,2024-10-24 10:13:57.952936+00:00,05780cb3-8795-4857-babe-83d38facf9d3,False,False,35
3,6b6afe41-ab0d-4b0e-8f39-44a18c80a10b,Friendly Greeting,2024-10-24 10:13:36.033454+00:00,6d08a2bc-eeec-4e11-8cc0-e32e125ad89e,assistant,I aim to be helpful while keeping responses f...,2024-10-24 10:13:57.952936+00:00,306bd439-1f26-485e-aff6-6c7ce689858f,False,False,277
4,0a9802a9-a708-4cd7-808e-63978115af62,React TypeScript Chrome Extension Boilerplate,2024-10-25 19:20:38.805074+00:00,5804db08-89ce-421f-a2ce-5a3f5c86ad68,human,Can you generate some boilerplate react ts vit...,2024-10-25 19:21:02.439211+00:00,00000000-0000-4000-8000-000000000000,False,False,75


## Register with DuckDB for SQL analysis

In [ ]:
con = duckdb.connect("data/ai_chat.duckdb", read_only=True)
con.execute("SHOW ALL TABLES").fetchall()

,sender,messages,avg_chars
0,human,1350,109
1,assistant,1348,1314


In [6]:
# conversations over time (by month)
con.execute("""
    SELECT
        DATE_TRUNC('month', conversation_created_at) AS month,
        COUNT(DISTINCT conversation_uuid) AS conversations
    FROM messages
    GROUP BY 1
    ORDER BY 1
""").df()

,month,conversations
0,2024-10-01 00:00:00+01:00,4
1,2024-11-01 00:00:00+00:00,1
2,2025-03-01 00:00:00+00:00,12
3,2025-04-01 00:00:00+01:00,15
4,2025-05-01 00:00:00+01:00,12
5,2025-06-01 00:00:00+01:00,14
6,2025-07-01 00:00:00+01:00,14
7,2025-08-01 00:00:00+01:00,2
8,2025-09-01 00:00:00+01:00,1
9,2026-01-01 00:00:00+00:00,16


In [7]:
# longest conversations by message count
con.execute("""
    SELECT
        conversation_name,
        COUNT(*) AS total_messages,
        MIN(created_at) AS started,
        MAX(created_at) AS last_message
    FROM messages
    GROUP BY conversation_uuid, conversation_name
    ORDER BY total_messages DESC
    LIMIT 10
""").df()

,conversation_name,total_messages,started,last_message
0,Men's casual office wear in Bolton,78,2026-04-18 14:52:30.625062+01:00,2026-04-18 18:24:08.256067+01:00
1,Customizing Android terminal colors in desktop...,64,2026-04-12 11:43:10.515465+01:00,2026-04-12 12:19:45.536298+01:00
2,,60,2026-05-26 02:23:53.759159+01:00,2026-05-26 03:28:19.623944+01:00
3,HSBC managed fund vs S&P 500,46,2026-04-03 03:02:54.779681+01:00,2026-04-03 03:59:12.558309+01:00
4,,44,2026-04-19 22:10:14.376888+01:00,2026-04-20 07:44:46.314816+01:00
5,"Choosing between dbt, Polars, Spark, and DuckDB",36,2026-05-11 20:58:07.531421+01:00,2026-05-11 21:22:05.496338+01:00
6,Personal data lake architecture,34,2026-04-03 16:42:15.709231+01:00,2026-04-03 17:40:34.405752+01:00
7,Isolating deno notebooks from npm packages,30,2026-01-14 21:10:38.306202+00:00,2026-01-14 21:57:44.936974+00:00
8,,30,2026-04-13 21:42:59.480097+01:00,2026-04-13 23:00:04.516795+01:00
9,,28,2026-05-16 19:53:18.832586+01:00,2026-05-16 21:58:18.992029+01:00


In [8]:
# activity by hour of day
con.execute("""
    SELECT
        EXTRACT(hour FROM created_at) AS hour,
        COUNT(*) AS messages
    FROM messages
    WHERE sender = 'human'
    GROUP BY 1
    ORDER BY 1
""").df()

,hour,messages
0,0,9
1,1,6
2,2,26
3,3,42
4,4,12
5,5,4
6,6,8
7,7,28
8,8,45
9,9,66
